# UNIQAT: single image assessment

This notebook is the Jupyter twin of `examples/single_image_assessment.py`. It walks through the minimum code needed to assess one underwater image with UNIQAT, inspect the composite scores, and generate a full visual quality report.

Prerequisites: clone https://github.com/open-AIMS/UNIQAT and install with `pip install -e .` (the base install is enough for this example, no deep learning extras required). Run the notebook from the repository root.

## 1. Imports and example image

UNIQAT exposes the assessor, the quality visualiser, and a one-line `assess_image` convenience function directly from the top-level package. The bundled example `examples/18_img_good.png` is a high-quality reef image that should score in the *Excellent* category.

In [ ]:
from pathlib import Path

from uniqat import UnderwaterImageAssessor, QualityVisualizer

EXAMPLE_IMAGE = Path('examples') / '18_img_good.png'
assert EXAMPLE_IMAGE.is_file(), 'Run this notebook from the UNIQAT repository root.'

## 2. Run the assessment

`UnderwaterImageAssessor.assess()` computes all 37 individual metrics and packs them into a `QualityAssessment` dataclass alongside the composite scores. The assessor reads the image once, so it is cheap to reuse across several calls for batch pipelines.

In [ ]:
assessor = UnderwaterImageAssessor(image_path=str(EXAMPLE_IMAGE))
assessment = assessor.assess()

print(f'Overall quality:            {assessment.overall_score:5.1f} / 100')
print(f'Usability category:         {assessment.usability_category}')
print(f'Feature usefulness:         {assessment.feature_usefulness:5.1f} / 100')
print(f'Marine science value:       {assessment.marine_science_value:5.1f} / 100')
print(f'Blue-water severity:        {assessment.blue_water_problem_severity:5.2f} / 10')

## 3. Inspect the detailed metrics

All 37 metrics live in `assessment.detailed_metrics` as a dictionary. The `to_dict()` method serialises the full assessment, including metrics and composite scores, so it can be written directly to JSON.

In [ ]:
detailed = assessment.detailed_metrics
print(f'Computed {len(detailed)} detailed metrics.')
for name in list(detailed)[:6]:
    value = detailed[name]
    formatted = f'{value:.3f}' if isinstance(value, (int, float)) else str(value)
    print(f'  {name}: {formatted}')

## 4. Generate the visual quality report

`QualityVisualizer.create_comprehensive_report(path)` writes a multi-panel PNG summarising colour channels, histograms, blue-water indicators, feature richness, and the overall composite score band.

In [ ]:
report_path = Path('examples') / '18_img_good_uniqat_report_notebook.png'
visualiser = QualityVisualizer(
    assessor.metrics_calculator.image,
    assessor.metrics,
    assessment.to_dict(),
)
visualiser.create_comprehensive_report(str(report_path))
print(f'Wrote visual report: {report_path}')

## 5. One-line alternative

For quick scripting or when you only need the composite scores, `uniqat.assess_image(path)` wraps steps 2 and 3 into a single call.

In [ ]:
import uniqat

quick = uniqat.assess_image(EXAMPLE_IMAGE)
print(f'Quick score: {quick.overall_score:.1f} ({quick.usability_category})')